In [1]:
import os
import warnings

# Нужен для загрузки данных и скриптов
os.chdir("/Data/EEG-Visual-Experiment")
warnings.filterwarnings('ignore')

### Методы группировки sample датасетов perception и imagery

#### Подгрузка пакетов и датасетов

In [2]:
import numpy as np
import pandas as pd

from Scripts.Data_Loader import EIRDataset

In [3]:
path_perception = "./Generated/Data_Pattern/"
path_imagery = "./Generated/Data_Train/"

ds_perception = EIRDataset(path_perception, task_type="geometric", n_jobs=70)  # Do not set the `n_jobs` = 72 (Can crash the server)
ds_imagery = EIRDataset(path_imagery, task_type="geometric", n_jobs=70)  # Do not set the `n_jobs` = 72 (Can crash the server)

Loading .fif files: 100%|██████████| 840/840 [00:32<00:00, 25.57it/s]


#### Создание ссылочных таблиц

In [15]:
ref_perception = pd.DataFrame([
    {
        "pdsi": idx,
        "subject_id": meta["subject_id"],
        "trial_id": meta["trial_id"],
        "pattern_id": pattern_id,
    }
    for idx, (_, _, meta, pattern_id, _) in enumerate(ds_perception)
])
ref_perception.head()

,pdsi,subject_id,trial_id,pattern_id
0,0,4,1,6
1,1,4,1,7
2,2,4,1,3
3,3,4,1,12
4,4,4,1,8


In [16]:
ref_imagery = pd.DataFrame([
    {
        "idsi": idx,
        "subject_id": meta["subject_id"],
        "trial_id": meta["trial_id"],
        "pattern_id": pattern_id,
    }
    for idx, (_, _, meta, pattern_id, _) in enumerate(ds_imagery)
])
ref_imagery.head()

,idsi,subject_id,trial_id,pattern_id
0,0,34,1,6
1,1,34,1,8
2,2,34,1,0
3,3,34,1,8
4,4,34,1,8


#### Слияние таблиц

In [39]:
keys = ["subject_id", "trial_id", "pattern_id"]

ref_grouped = (
    ref_perception
    .merge(
        ref_imagery.groupby(keys, as_index=False)["idsi"].agg(list),
        on=keys,
        how="left",
    )
    .rename(columns={
        "pdsi": "perception_dataset_index",
        "idsi": "imagery_dataset_indexes"
    })
)

ref_grouped["imagery_dataset_indexes"] = ref_grouped["imagery_dataset_indexes"].apply(lambda x: x if isinstance(x, list) else [])

ref_grouped.head()

,perception_dataset_index,subject_id,trial_id,pattern_id,imagery_dataset_indexes
0,0,4,1,6,"[182, 185, 186, 187]"
1,1,4,1,7,"[184, 188]"
2,2,4,1,3,[183]
3,3,4,1,12,"[189, 192, 193, 194]"
4,4,4,1,8,"[191, 195]"


In [40]:
ref_grouped.to_csv("./Generated/Results/perception_imagery_grouped.csv", index=False)

#### Аналитическое сравнения отношение subject к pattern

In [89]:
df = ref_grouped.groupby(["subject_id", "pattern_id"], as_index=False)[["perception_dataset_index", "trial_id", "imagery_dataset_indexes"]].agg(list)
df[df["perception_dataset_index"].str.len() > 1]\
    .groupby(["pattern_id"], as_index=False)\
    .agg(
        imagery_dataset_indexes=("imagery_dataset_indexes", len),
    )

,pattern_id,imagery_dataset_indexes
0,0,6
1,1,4
2,2,5
3,3,4
4,4,8
5,5,6
6,6,5
7,7,6
8,8,4
9,9,8


In [90]:
df[df["perception_dataset_index"].str.len() > 1]\
    .groupby(["subject_id"], as_index=False)\
    .agg(
        pattern_id=("pattern_id", list),
    )

,subject_id,pattern_id
0,1,"[9, 11]"
1,2,"[9, 10]"
2,3,"[0, 3, 11, 12]"
3,4,"[6, 7, 12]"
4,5,"[3, 8, 9, 10]"
5,6,"[4, 7, 8, 9]"
6,7,"[4, 5, 8]"
7,8,"[3, 7, 10]"
8,9,"[5, 6]"
9,10,"[5, 8]"


#### Универсальная функция для создания объедененной таблицы отношения Perception к Imagery

In [91]:
def create_perception_imagery_table(
    perception: EIRDataset,
    imagery: EIRDataset
) -> pd.DataFrame:
    ref_perception = pd.DataFrame([
        {
            "pdsi": idx,
            "subject_id": meta["subject_id"],
            "trial_id": meta["trial_id"],
            "pattern_id": pattern_id,
        }
        for idx, (_, _, meta, pattern_id, _) in enumerate(perception)
    ])
    ref_imagery = pd.DataFrame([
        {
            "idsi": idx,
            "subject_id": meta["subject_id"],
            "trial_id": meta["trial_id"],
            "pattern_id": pattern_id,
        }
        for idx, (_, _, meta, pattern_id, _) in enumerate(ds_imagery)
    ])
    
    keys = ["subject_id", "trial_id", "pattern_id"]

    ref_grouped = (
        ref_perception
        .merge(
            ref_imagery.groupby(keys, as_index=False)["idsi"].agg(list),
            on=keys,
            how="left",
        )
        .rename(columns={
            "pdsi": "perception_dataset_index",
            "idsi": "imagery_dataset_indexes"
        })
    )
    
    ref_grouped["imagery_dataset_indexes"] = ref_grouped["imagery_dataset_indexes"].apply(lambda x: x if isinstance(x, list) else [])

    return ref_grouped.copy()

In [92]:
create_perception_imagery_table(ds_perception, ds_imagery).head()

,perception_dataset_index,subject_id,trial_id,pattern_id,imagery_dataset_indexes
0,0,4,1,6,"[182, 185, 186, 187]"
1,1,4,1,7,"[184, 188]"
2,2,4,1,3,[183]
3,3,4,1,12,"[189, 192, 193, 194]"
4,4,4,1,8,"[191, 195]"
